In [12]:
import torchvision
from pandas.core.ops.mask_ops import raise_for_nan
from sympy.printing.pretty.pretty_symbology import line_width
from torch.onnx.ops import attention
from torchvision import transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
tr_ds = torchvision.datasets.CIFAR10(root='../data', train=True, download=True, transform=transform)
tt_ds = torchvision.datasets.CIFAR10(root='../data', train=False, download=True, transform=transform)

In [13]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.mps.is_available() else 'cpu')

In [14]:
# 초기화
def trunc_normal_(t, mean=0.0, std=1.0):
    with torch.no_grad():
        size = t.shape
        tmp = t.new_empty(size+(4, )).normal_()
        valid = (tmp<2)&(tmp>-2)
        ind = valid.max(-1, keepdim=True)[1]
        t.data.copy_(tmp.gather(-1, ind).squeeze(-1))
        t.data.mul_(std).add_(mean)
        return t

In [25]:
# 입력 -> (B,C,H,W) | 출력 -> (B,N,D)
# N -> (H/ps) * (W\ps)
class PatchEmbed(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=768):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.grid_size = self.img_size//self.patch_size
        self.num_patches = self.grid_size*self.grid_size
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x): # 패치분할 + 선형투영
        x = self.proj(x) # 패치분할(B,D,N)
        x = x.flatten(2).transpose(1,2) # 선형투영 (B,N,D)
        return x

class MLP(nn.Module): # 토큰별 비선형 변환
    def __init__(self, dim, mlp_ratio=4.0, drop=0.0):
        super().__init__()
        hidden = int(dim * mlp_ratio)
        self.fc1 = nn.Linear(dim, hidden)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden, dim)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x

# 멀티헤드 어텐션
class Attention(nn.Module):
    def __init__(self, dim, num_heads=12, qkv_bias=True, attn_drop=0.0, proj_drop=0.0):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim//num_heads
        self.scale = self.head_dim**-0.5

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

        self.last_attn = None # (B,heads,N,N)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        q,k,v = qkv.unbind(dim=2)
        q = q.permute(0, 2, 1, 3) # (B,heads,N,d)
        k = k.permute(0, 2, 1, 3) # (B,heads,N,d)
        v = v.permute(0, 2, 1, 3) # (B,heads,N,d)

        attn = (q@ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        self.last_attn = attn

        x = attn@v
        x = x.transpose(1, 2).reshape(B, N, -1)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

class Block(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4.0, qkv_bias=True, drop=0.0, attn_drop=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, eps=1e-6)
        self.attn = Attention(dim, num_heads, qkv_bias, attn_drop, drop)
        self.norm2 = nn.LayerNorm(dim, eps=1e-6)
        self.mlp = MLP(dim, mlp_ratio, drop)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

In [26]:
class ViT(nn.Module):
    def __init__(self,
                 img_size=224,
                 patch_size=16,
                 in_chans=3,
                 num_classes=1000,
                 embed_dim=768,
                 depth=12,
                 num_heads=12,
                 mlp_ratio=4.0,
                 drop_rate=0.0,
                 attn_drop_rate=0.0):
        super().__init__()
        self.num_classes = num_classes
        self.num_features = embed_dim

        self.patch_embed = PatchEmbed(img_size, patch_size, in_chans, embed_dim)
        num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches+1, embed_dim))
        self.pos_drop = nn.Dropout(drop_rate)

        self.blocks = nn.ModuleList([Block(embed_dim, num_heads, mlp_ratio, qkv_bias=True, drop=drop_rate, attn_drop=attn_drop_rate) for _ in range(depth)])

        self.norm = nn.LayerNorm(embed_dim, eps=1e-6)
        self.head = nn.Linear(embed_dim, num_classes)

        self.apply(self._init_weights)
        trunc_normal_(self.pos_embed, std=0.02)
        trunc_normal_(self.cls_token, std=0.02)


    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode='fan_out')
            if m.bias is not None:
                nn.init.zeros_(m.bias)

    def forward_features(self, x):
        B = x.size(0)
        x = self.patch_embed(x)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x = x + self.pos_embed
        x = self.pos_drop(x)
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        return x[:,0]

    def forward(self, x):
        x = self.forward_features(x)
        x = self.head(x)
        return x

def vit_tiny(img_size=224, patch_size=16, num_classes=10):
    return ViT(img_size, patch_size, embed_dim=192, depth=12, num_heads=3, num_classes=num_classes)

In [27]:
import torchvision
from torchvision import transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
def get_ds_loader():
    tr_ds = torchvision.datasets.CIFAR10(root='../data', train=True, download=True, transform=transform)
    tt_ds = torchvision.datasets.CIFAR10(root='../data', train=False, download=True, transform=transform)
    tr_ds_loader = torch.utils.data.DataLoader(tr_ds, batch_size=16, shuffle=True, num_workers=4, pin_memory=torch.cuda.is_available())
    tt_ds_loader = torch.utils.data.DataLoader(tt_ds, batch_size=16, shuffle=False, num_workers=4, pin_memory=torch.cuda.is_available())
    return tr_ds, tt_ds, tr_ds_loader, tt_ds_loader

In [28]:
tr_ds, tt_ds, tr_ds_loader, tt_ds_loader = get_ds_loader()

In [29]:
img, label = next(iter(tt_ds_loader))
img, label = img.to(DEVICE), label.to(DEVICE)
img.shape, label.shape

(torch.Size([16, 3, 224, 224]), torch.Size([16]))

In [35]:
m = vit_tiny(num_classes=10).to(DEVICE)
m.eval()

from collections import defaultdict
activations = {}
attentions = defaultdict(list)

def save_activation(name):
    def hook(m, i, o):
        activations[name] = o.detach().cpu()
    return hook

def save_attn(block_idx):
    def hook(m, i, o):
        if hasattr(m.attn,'last_attn') and m.attn.last_attn is not None:
            attentions[block_idx].append(m.attn.last_attn.detach().cpu())
    return hook

h_hooks = []
h_hooks.append(m.patch_embed.register_forward_hook(save_activation('patch_embed_out')))
for bi, blk in enumerate(m.blocks):
    h_hooks.append(blk.register_forward_hook(save_attn(bi)))
with torch.no_grad():
    _ = m(img)
print(f'activations.keys: {activations.keys()}')
print(f'attentions.keys: {attentions.keys()}')

activations.keys: dict_keys(['patch_embed_out'])
attentions.keys: dict_keys([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11])


In [41]:
for i in range(16):
    print(attentions[i][0].shape)

torch.Size([16, 3, 197, 197])
torch.Size([16, 3, 197, 197])
torch.Size([16, 3, 197, 197])
torch.Size([16, 3, 197, 197])
torch.Size([16, 3, 197, 197])
torch.Size([16, 3, 197, 197])
torch.Size([16, 3, 197, 197])
torch.Size([16, 3, 197, 197])
torch.Size([16, 3, 197, 197])
torch.Size([16, 3, 197, 197])
torch.Size([16, 3, 197, 197])
torch.Size([16, 3, 197, 197])


IndexError: list index out of range

In [ ]:
def denorm_for_vit(img):
    x = img.detach().cpu()
    x = ((x-x.min())/(x.max()-x.min()) + 1e-6)
    return x

def show_img_with_patches(img, patch_size=16):
    x = denorm_for_vit(img).permute(1, 2, 0).numpy()
    H, W =x.shape[:2]
    plt.figure()
    plt.imshow(x)
    for r in range(0, H, patch_size):
        plt.axhline(r, linewidth=0.5)
    for c in range(0, W, patch_size):
        plt.axhline(c, linewidth=0.5)
    plt.title('grid')
    plt.axis('off')
    plt.show()

def pca_project_to_rgb(tokens):
    x = tokens-tokens.mean(0, keepdim=True)
    U, S, Vt = np.linalg.svd(X, full_matrices=False)